<center><h1>Feature Engineering</h1>
After looking at the quality of the data through exploratory data analysis, we will apply data cleaning and processing.</center>

## Import package and Load Processed Data

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('df_processed.csv')

Separate the features based on defined categories. This a crucial step because variables like `firstbloodassist` usually have missing values because the database includes team stats. They should be treated separately. Furthermore, player stats will be engineered into different features. Identifiers like `playername` and `teamname` will be removed, as well as the drafting as we will be focusing on team strength. The feature `champions` will be turned into a rolling win_rate of that champion.

In [4]:
player_cols = ['gameid', 'date', 'position', 'teamid', 'champion', 'doublekills', 'triplekills', 'quadrakills', 'pentakills', 
               'firstbloodkill', 'firstbloodassist', 'firstbloodvictim', 'visionscore', 'kills', 'assists', 'dpm', 
               'totalgold', 'killsat10', 'assistsat10', 'opp_killsat10', 'opp_deathsat10', 'killsat15', 'assistsat15', 
               'opp_killsat15', 'killsat20', 'assistsat20', 'opp_killsat20', 'killsat25', 'assistsat25', 'opp_killsat25', 
               'opp_deathsat25', 'golddiffat10', 'xpdiffat10', 'golddiffat15', 'xpdiffat15', 'golddiffat25']
team_cols = ['gameid', 'league', 'split', 'date' 'side', 'teamid', 'result', 'team kpm', 'firstdragon', 'dragons',
             'opp_dragons', 'elders', 'opp_elders', 'firstherald', 'void_grubs', 'opp_void_grubs', 'firstbaron', 
             'barons', 'opp_barons', 'firsttower', 'towers', 'opp_towers']

## *"diff"* Features
A team can win or loss whether they have 50 kills or 20 kills. What matters is their kill advantage over the opponent. Just like with `golddiffatXX` or `xpdiffatXX`, we're going to create `killsdiffatXX` and `baronsdiff` features

In [5]:
# Get the opp features since they surely have a pair
opp_features = [col for col in df.columns if col[:3] == 'opp']
print("opp_features:", opp_features)

# Create the diff features by subracting opp stats to team stats
for opp in opp_features:
    team = opp[4:]
    print(f"\nCreating {team}diff feature from the difference between {team} and {opp} features...")
    df[f'{team}diff'] = df[f'{team}'] - df[f'{opp}']
    print(f"Dropping {team} and {opp} features...")
    df = df.drop([f'{team}', f'{opp}'], axis='columns', errors="ignore")

opp_features: ['opp_dragons', 'opp_elders', 'opp_void_grubs', 'opp_barons', 'opp_towers', 'opp_killsat10', 'opp_killsat15', 'opp_killsat20', 'opp_killsat25']

Creating dragonsdiff feature from the difference between dragons and opp_dragons features...
Dropping dragons and opp_dragons features...

Creating eldersdiff feature from the difference between elders and opp_elders features...
Dropping elders and opp_elders features...

Creating void_grubsdiff feature from the difference between void_grubs and opp_void_grubs features...
Dropping void_grubs and opp_void_grubs features...

Creating baronsdiff feature from the difference between barons and opp_barons features...
Dropping barons and opp_barons features...

Creating towersdiff feature from the difference between towers and opp_towers features...
Dropping towers and opp_towers features...

Creating killsat10diff feature from the difference between killsat10 and opp_killsat10 features...
Dropping killsat10 and opp_killsat10 features..

## Z-score Transformation
Raw stats that does not have any opponent stats to compare to does not provide any meaningful insight. Applying z-score will standardize and compare each performance against the others.

In [8]:
df.columns

Index(['gameid', 'league', 'split', 'date', 'side', 'position', 'playername',
       'teamname', 'teamid', 'firstPick', 'champion', 'ban1', 'ban2', 'ban3',
       'ban4', 'ban5', 'doublekills', 'triplekills', 'quadrakills',
       'pentakills', 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim',
       'visionscore', 'result', 'team kpm', 'firstdragon', 'firstherald',
       'firstbaron', 'firsttower', 'kills', 'assists', 'dpm', 'totalgold',
       'assistsat10', 'assistsat15', 'assistsat20', 'assistsat25',
       'golddiffat10', 'xpdiffat10', 'xpdiffat15', 'golddiffat25',
       'dragonsdiff', 'eldersdiff', 'void_grubsdiff', 'baronsdiff',
       'towersdiff', 'killsat10diff', 'killsat15diff', 'killsat20diff',
       'killsat25diff'],
      dtype='str')

In [9]:
raw_stats = ['visionscore', 'kills', 'assists', 'dpm', 'totalgold',
            'assistsat10', 'assistsat15', 'assistsat20', 'assistsat25']

In [10]:
# Split the dataframe into team-specific stats and player-specific stats
df_team = df[df["position"] == "team"].copy()
df_player = df[df["position"] != "team"].copy()

## Rolling Metric
Since all numerical features, like `dpm` or `kills`, are data we will only know after the game had concluded, there will be data leakage if we use same-day game stats to predict the same game result. To circumvent data leakage, we will implement a rolling 5-step average for each metric, with a minimum of 3-step window for teams that has smaller historical data.

In [ ]:
# Sort chronologically
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['date']).reset_index(drop=True)

In [ ]:
# Calculate rolling past match averages grouped by team
cols_to_be_rolled = df.select_dtypes(include=["number"]).columns
num_matches = 5

for col in cols_to_be_rolled:
  col_name = f"{col}_rolling_avg_{num_matches}"
  # Group by team and calculate rolling mean, shifting by 1 so we don't leak the current match stats
  df[col_name] = (
      df.groupby("teamname")[col]
      .shift(1)
      .rolling(window=num_matches, min_periods=3)
      .mean()
  )

In [ ]:
# Drop rows where rolling features are NaN (i.e., each team's first match)
df = df.dropna().reset_index(drop=True)

## Removing Non-Diff Features
Absolute team stats would create noise and may mislead the model. The features that has no corresponding opp stats are leftovers.

In [ ]:
non_diff = ['kills', 'deaths', 'dpm', 'damagetakenperminute', 'wardskilled_rolling', 'totalgold']

for col in non_diff:
    df = df.drop([f'{col}_rolling_avg_5',], axis='columns', errors="ignore")

## Data Readiness
The data is almost ready for modeling. We will just have to remove the identifiers like `gameid` and `date`, and drop same-day features like `kills` and `deaths`. We only need the rolling averages for modeling. Finally, we will encode `split` for chronological validation splitting during modeling.

In [ ]:
# Drop identifiers
df = df.drop(['gameid', 'league', 'date', 'teamname'], axis='columns', errors='ignore')

In [ ]:
cols_to_drop = [x for x in cols_to_be_rolled if x != 'result']

# Dropping same-day features
df = df.drop(cols_to_drop+['split'], axis='columns', errors='ignore')

In [ ]:
# # One-Hot Enconde split and dropping the first column to remove multicollinearity
# df = pd.get_dummies(df, columns=['split'], drop_first=True)

In [ ]:
# Compute correlation matrix
co_mtx = df.corr(numeric_only=True)

# Plot correlation heatmap
plt.figure(figsize=(15, 15))
sns.heatmap(co_mtx, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('CORRELATION HEATMAP', fontsize=16, fontweight="bold")
plt.show()

In [ ]:
# Quick snapshot of the dataframe ready for modeling
df.info()

In [ ]:
df.to_csv("df_ready.csv", index=False)